# Desafio Gerdau – Diagnóstico da base de códigos de serviço

Notebook com o diagnóstico (EDA) da base anonimizada. Rode as células de cima para baixo.

**Etapas**
1. Carregar e conhecer a base
2. Tamanho e qualidade
3. Repetição (3 definições)
4. Nomes confusos (flags)
4b. Validação da unidade de medida (abas 2 e 3 do dataset)
5. Exportar resultados

### Imports

In [ ]:
%pip install pandas

In [22]:
import re
import unicodedata
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

## 1. Carregando e conhecendo a base

In [23]:
ARQUIVO = './content/dataset_gerdau.xlsx'  
ABA = 0                        # 0 = primeira aba (Base Anonimizada)

df = pd.read_excel(ARQUIVO, sheet_name=ABA)
df = df.iloc[:, :5]            # garante só as 5 colunas originais
df.columns = ['grupo', 'fornecedor', 'codigo', 'descricao', 'um']

print(df.shape)
print(df.dtypes)
df.head(10)

(9795, 5)
grupo           str
fornecedor    int64
codigo        int64
descricao       str
um              str
dtype: object


,grupo,fornecedor,codigo,descricao,um
0,Civil,200982674,4033776,SV-EQUIPE TELHADO,SV
1,Civil,201464634,4012475,DIARIA RESGATISTA NIVEL 3 ATE 8H .,SV
2,Civil,200772986,4050934,SV OFICIAL MENSAL,SV
3,Civil,300623388,4097173,Pavimentação a frio por kg,KG
4,Civil,200982674,4066840,SV-EQUIPE PINTURA,SV
5,Civil,200772986,4050928,SV ENCARREGADO MENSAL,SV
6,Civil,201332350,4096183,ELETRICISTA - ROTINA,SV
7,Civil,200772986,4050925,SV ENGENHEIRO MENSAL,SV
8,Civil,300056198,4023627,SV-HORA OFICIAL-HR NORMAL,H
9,Civil,201464634,4017177,Diaria resgatista turno de ate 8 horas .,SV


## 2. Tamanho e qualidade

Cada linha é a relação **código de serviço × fornecedor**, então linhas ≠ códigos distintos.

In [24]:
print('Linhas:', len(df))
print('Códigos de serviço distintos:', df['codigo'].nunique())
print('Fornecedores distintos:', df['fornecedor'].nunique())
print('\nNulos por coluna:')
print(df.isna().sum())

Linhas: 9795
Códigos de serviço distintos: 7937
Fornecedores distintos: 111

Nulos por coluna:
grupo         0
fornecedor    0
codigo        0
descricao     0
um            0
dtype: int64


In [25]:
resumo_grupo = df.groupby('grupo').agg(
    linhas=('codigo', 'size'),
    codigos=('codigo', 'nunique'),
    fornecedores=('fornecedor', 'nunique'))
resumo_grupo

,linhas,codigos,fornecedores
grupo,,,
Civil,4558,3451,24
Eletromecânica,2456,1779,29
Facilities,2781,2730,62


In [26]:
print('Unidades de medida:')
print(df['um'].value_counts())

Unidades de medida:
um
SV     4083
H      3152
ST     1039
M2      571
M       444
M3      159
MON     139
KG       71
CM       47
TAG      37
ZCJ      16
KM       15
L         8
CDM       6
MM        4
TO        2
ML        2
Name: count, dtype: int64


## 3. Normalização

In [27]:
def norm(s):
    """Maiúsculas, sem acento, sem pontuação e sem espaços extras."""

    s = (
        unicodedata.normalize('NFKD', str(s))
        .encode('ascii', 'ignore')
        .decode()
        .upper()
    )

    s = s.replace('/', ' ')
    s = s.replace('+', ' + ')

    s = re.sub(r'[-_.,;:()"]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()

    return s


# Descrição original em maiúsculas
df['descricao_lit'] = (
    df['descricao']
    .astype(str)
    .str.upper()
    .str.strip()
)

# Descrição normalizada
df['descricao_norm'] = df['descricao'].map(norm)

# Conferir 10 exemplos
df[['descricao', 'descricao_lit', 'descricao_norm']].sample(
    10,
    random_state=1
)


,descricao,descricao_lit,descricao_norm
2573,LÍDER ELÉTRICA - HORA INSALUBRE CHEIA,LÍDER ELÉTRICA - HORA INSALUBRE CHEIA,LIDER ELETRICA HORA INSALUBRE CHEIA
7632,SV-AUXILIAR SERVIÇOS GERAIS-ADM,SV-AUXILIAR SERVIÇOS GERAIS-ADM,SV AUXILIAR SERVICOS GERAIS ADM
1755,SV VIDRO LISO 3 MILÍMETROS,SV VIDRO LISO 3 MILÍMETROS,SV VIDRO LISO 3 MILIMETROS
7759,H.E.SS-Eletricista NR35-ADM,H.E.SS-ELETRICISTA NR35-ADM,H E SS ELETRICISTA NR35 ADM
6964,MONTADOR(A) LIDER(HE NOT DOM.FER),MONTADOR(A) LIDER(HE NOT DOM.FER),MONTADOR A LIDER HE NOT DOM FER
5701,PREVENTIVA PONTES - 2415 - USB1,PREVENTIVA PONTES - 2415 - USB1,PREVENTIVA PONTES 2415 USB1
5880,SOLDADOR MAÇARIQUEIRO (HORA NOT),SOLDADOR MAÇARIQUEIRO (HORA NOT),SOLDADOR MACARIQUEIRO HORA NOT
7652,VIGILANTE DESARMADO 24H (SEG a DOM),VIGILANTE DESARMADO 24H (SEG A DOM),VIGILANTE DESARMADO 24H SEG A DOM
7834,H.E.SS-Sup.Áreas Verdes+mot.caminhão-ADM,H.E.SS-SUP.ÁREAS VERDES+MOT.CAMINHÃO-ADM,H E SS SUP AREAS VERDES + MOT CAMINHAO ADM
234,SV-OFICIAL COM MARTELO ROMPEDOR OU ELÉTR,SV-OFICIAL COM MARTELO ROMPEDOR OU ELÉTR,SV OFICIAL COM MARTELO ROMPEDOR OU ELETR


In [28]:

# 1. Linhas totais
linhas_totais = len(df)

# 2. Descrições brutas distintas
descricoes_brutas = df['descricao'].nunique()

# 3. Descrições normalizadas distintas
descricoes_norm = df['descricao_norm'].nunique()

# 4. Chaves distintas: descrição + grupo + unidade
colunas_chave = ['descricao_norm', 'grupo', 'um']

chaves_distintas = df[colunas_chave].drop_duplicates().shape[0]

# 5. Linhas que fazem parte de duplicações
duplicadas = df.duplicated(
    subset=colunas_chave,
    keep=False
)

linhas_duplicadas = int(duplicadas.sum())

# Exibir resultados
print('Linhas totais:', linhas_totais)
print('Descrições brutas distintas:', descricoes_brutas)
print('Descrições normalizadas distintas:', descricoes_norm)
print('Chaves distintas:', chaves_distintas)
print('Linhas duplicadas:', linhas_duplicadas)


Linhas totais: 9795
Descrições brutas distintas: 7468
Descrições normalizadas distintas: 7333
Chaves distintas: 7397
Linhas duplicadas: 3885


## 4. Repetição

"Repetido" tem três significados. Medimos um de cada vez:

- **4a.** Mesmo código em várias linhas.
- **4b.** Mesma descrição em códigos diferentes.
- **4c.** Mesmo significado com descrições diferentes.

### 4a. Mesmo código em mais de uma linha

In [11]:
vc = df['codigo'].value_counts()
print('Códigos que aparecem em mais de uma linha:', (vc > 1).sum())
print('Máximo de repetições de um mesmo código:', vc.max())

# o mesmo código tem fornecedores diferentes? (esperado)
rep = df[df['codigo'].duplicated(keep=False)].sort_values('codigo')
rep.head(20)

Códigos que aparecem em mais de uma linha: 1234
Máximo de repetições de um mesmo código: 6


,grupo,fornecedor,codigo,descricao,um,descricao_lit,descricao_norm
739,Civil,201341488,4035891,SV-CALDEIRARIA(MENSAL DEMANDA ),SV,SV-CALDEIRARIA(MENSAL DEMANDA ),SV CALDEIRARIA MENSAL DEMANDA
4574,Eletromecânica,200876404,4035891,SV-CALDEIRARIA(MENSAL DEMANDA ),SV,SV-CALDEIRARIA(MENSAL DEMANDA ),SV CALDEIRARIA MENSAL DEMANDA
4575,Eletromecânica,300805670,4035891,SV-CALDEIRARIA(MENSAL DEMANDA ),SV,SV-CALDEIRARIA(MENSAL DEMANDA ),SV CALDEIRARIA MENSAL DEMANDA
740,Civil,201341488,4035892,SV-CALDEIRARIA(QUINZENAL DEMANDA),SV,SV-CALDEIRARIA(QUINZENAL DEMANDA),SV CALDEIRARIA QUINZENAL DEMANDA
4577,Eletromecânica,300805670,4035892,SV-CALDEIRARIA(QUINZENAL DEMANDA),SV,SV-CALDEIRARIA(QUINZENAL DEMANDA),SV CALDEIRARIA QUINZENAL DEMANDA
4576,Eletromecânica,200876404,4035892,SV-CALDEIRARIA(QUINZENAL DEMANDA),SV,SV-CALDEIRARIA(QUINZENAL DEMANDA),SV CALDEIRARIA QUINZENAL DEMANDA
4578,Eletromecânica,200876404,4035893,SV-CALDEIRARIA(SEMANAL DEMANDA),SV,SV-CALDEIRARIA(SEMANAL DEMANDA),SV CALDEIRARIA SEMANAL DEMANDA
741,Civil,201341488,4035893,SV-CALDEIRARIA(SEMANAL DEMANDA),SV,SV-CALDEIRARIA(SEMANAL DEMANDA),SV CALDEIRARIA SEMANAL DEMANDA
4579,Eletromecânica,300805670,4035893,SV-CALDEIRARIA(SEMANAL DEMANDA),SV,SV-CALDEIRARIA(SEMANAL DEMANDA),SV CALDEIRARIA SEMANAL DEMANDA
4581,Eletromecânica,300805670,4035894,SV-CALDEIRARIA(DIARIA DEMANDA),SV,SV-CALDEIRARIA(DIARIA DEMANDA),SV CALDEIRARIA DIARIA DEMANDA


### 4b. Mesma descrição em códigos diferentes

In [29]:
for col, rotulo in [('descricao_lit', 'literal (só maiúsculas)'),
                    ('descricao_norm', 'normalizada')]:
    g = df.groupby(col)['codigo'].nunique()
    print(f'Descrições distintas ({rotulo}):', g.shape[0])
    print(f'  com mais de um código:', (g > 1).sum())
    print(f'  códigos envolvidos:', df[df[col].isin(g[g > 1].index)]['codigo'].nunique())
    print()

Descrições distintas (literal (só maiúsculas)): 7468
  com mais de um código: 375
  códigos envolvidos: 845

Descrições distintas (normalizada): 7333
  com mais de um código: 486
  códigos envolvidos: 1091



In [30]:
# Classificação de cada linha pela descrição normalizada
qtd_linhas = df.groupby('descricao_norm')['codigo'].transform('size')
qtd_mesmo_forn = df.groupby(['descricao_norm', 'fornecedor'])['codigo'].transform('size')

df['situacao'] = 'Único'
df.loc[qtd_linhas > 1, 'situacao'] = 'Repetido entre fornecedores'
df.loc[(qtd_linhas > 1) & (qtd_linhas == qtd_mesmo_forn), 'situacao'] = 'Repetido no mesmo fornecedor'

print(df['situacao'].value_counts())
print()
print(pd.crosstab(df['grupo'], df['situacao']))

situacao
Único                           5830
Repetido entre fornecedores     3447
Repetido no mesmo fornecedor     518
Name: count, dtype: int64

situacao        Repetido entre fornecedores  Repetido no mesmo fornecedor  Único
grupo                                                                           
Civil                                  2125                           138   2295
Eletromecânica                         1182                            67   1207
Facilities                              140                           313   2328


## 5. Nomes confusos (flags)

Cada sinal vira uma coluna `f_...` (True/False). São **aproximações**: revise amostras antes de reportar.

In [31]:
d = df['descricao'].astype(str)

# 1) Provável truncamento (limite de 40 caracteres do SAP)
df['f_limite40'] = d.str.len() >= 40

# 2) Candidata a genérica (2 palavras ou menos, depois de normalizar)
df['f_generica'] = df['descricao_norm'].str.split().str.len() <= 2

# 3) Código ou contrato dentro do nome
padrao_cod = r'\bCOD\.?\s*\d+|\bCONTR(?:ATO)?\.?\s*\d+|\bCT\s*\d+'
df['f_cod_no_nome'] = d.str.contains(padrao_cod, case=False, regex=True)

# 4) Tipo de prefixo
def prefixo(s):
    s = s.upper().strip()
    if re.match(r'^SV\s*-', s): return 'SV-'
    if re.match(r'^SV\s', s):   return 'SV '
    if re.match(r'^H\s*-', s):  return 'H-'
    return 'sem prefixo'
df['prefixo'] = d.map(prefixo)

# 5) Sujeira de formatação
df['f_sujeira'] = (d != d.str.strip()) | d.str.contains(r'\s{2,}') | d.str.endswith('.')

In [32]:
flags = [c for c in df.columns if c.startswith('f_')]

resumo = df[flags].sum().to_frame('itens')
resumo['%'] = (resumo['itens'] / len(df) * 100).round(1)
display(resumo)

print('Prefixos:')
display(df['prefixo'].value_counts())

print('Onde cada problema se concentra (proporção por grupo):')
display(df.groupby('grupo')[flags].mean().round(3))

,itens,%
f_limite40,1519,15.5
f_generica,426,4.3
f_cod_no_nome,1,0.0
f_sujeira,559,5.7


Prefixos:


prefixo
sem prefixo    5607
SV             2188
SV-            1898
H-              102
Name: count, dtype: int64

Onde cada problema se concentra (proporção por grupo):


,f_limite40,f_generica,f_cod_no_nome,f_sujeira
grupo,,,,
Civil,0.200,0.019,0.0,0.046
Eletromecânica,0.105,0.027,0.0,0.042
Facilities,0.125,0.098,0.0,0.089


### 5b. Validação da unidade de medida (usa as abas 2 e 3 do dataset)

A aba **3. Mão de Obra** lista os 13 modificadores de custo e a aba **2. Unidades de Medida** diz o que cada sigla significa. A regra: identificar o modificador de cada item de mão de obra e comparar a unidade cadastrada com a esperada.

| Modificador | Unidade esperada |
| --- | --- |
| Hora normal, hora extra, adicional noturno, hora extra noturna, periculosidade (hora), insalubridade | H (Hora) |
| Periculosidade (mensal), posto mensal | MON (Mês) |
| Mobilização/desmobilização | SV (Serviço) |

In [33]:
# Abas de referência (mostrar para conferir)
un  = pd.read_excel(ARQUIVO, sheet_name='2. Unidades de Medida')
mo  = pd.read_excel(ARQUIVO, sheet_name='3. Mão de Obra')
display(un)
display(mo)

# Todas as unidades da base existem na aba de unidades?
print('Unidades da base fora da aba 2:', set(df['um']) - set(un.iloc[:, 0]))

,Sigla Unidade de Medida,Descrição
0,CM,Centímetro
1,CDM,Centímetro cúbico
2,ZCJ,Conjunto
3,TAG,Dia
4,H,Hora
5,L,Litro
6,MON,Mês
7,M,Metro
8,M3,Metro cúbico
9,M2,Metro quadrado


,Especificidades Mão de Obra,Descrição
0,Hora normal,"Hora de trabalho em jornada padrão, sem nenhum adicional aplicado — a hora ""..."
1,Hora extra Seg a Sab,"Hora trabalhada além da jornada normal em dia de semana ou sábado, sujeita a..."
2,Hora extra Dom/Feriado,"Hora trabalhada além da jornada normal em domingo ou feriado, geralmente com..."
3,Adicional noturno (hora),Acréscimo pago sobre a hora normal quando o trabalho ocorre no período notur...
4,Hora extra noturna Seg a Sab,Combinação de hora extra (dia de semana/sábado) com o adicional noturno — pa...
5,Hora extra noturna Dom/Feriado,Combinação de hora extra em domingo/feriado com o adicional noturno — o cená...
6,Adicional periculosidade (hora),Acréscimo pago por hora trabalhada em atividade classificada como perigosa (...
7,Adicional periculosidade s/ hora extra,"Mesmo adicional de periculosidade, mas aplicado especificamente sobre horas ..."
8,Adicional periculosidade (mensal),"Mesmo adicional de periculosidade, mas cobrado como valor fixo mensal em vez..."
9,Adicional insalubridade (hora),Acréscimo pago por hora quando a atividade expõe o trabalhador a agentes noc...


Unidades da base fora da aba 2: set()


In [34]:
def up(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().upper()
    return re.sub(r'\s+', ' ', s).strip()

df['t'] = df['descricao'].map(up)

# modificadores (do mais específico ao mais geral) - baseados na aba 3
MODS = [
 ('HEN-DF', r'(HE|HX|H\.?E|EXTRA).*NOT.*(DOM|FER)|NOT.*(DOM|FER).*(HE|EXTRA)'),
 ('HEN-SS', r'(HE|HX|EXTRA).*NOT|NOT.*(HE|EXTRA)'),
 ('HE-DF',  r'(HE|HX|H\.E|EXTRA|100%).*(DOM|FER)|(DOM|FER).*(HE|EXTRA)'),
 ('HE-SS',  r'\b(HE|HX|H\.E\.?|HORA EXTRA|HR\.? EXTRAS?)\b|EXTRA'),
 ('AD-NOT', r'\bNOT\b|NOTURN|AD\.? ?NOT|ADC\.?NOT'),
 ('PER-M',  r'(PERIC|PERI|PRERIC).*(MES|MENS)'),
 ('PER-H',  r'PERIC|PRERIC'),
 ('INS-H',  r'INSAL'),
 ('POSTO-M',r'MENSAL|\bMES\b|POSTO'),
 ('MOB',    r'MOB'),
 ('RETRO',  r'RETROATIV'),
 ('HN',     r'HORA NORMAL|\bHN\b|HR\.? NORMAL|NORM'),
]
CARGOS = r'SOLDA|MACARIQ|CALDEIR|ELETRIC|ELET\b|ELETR|MECANIC|MEC\b|PINTOR|ENCANAD|HIDRAULIC|MONTADOR|PEDREIR|CARPINT|SERRALH|AJUD|AUX DE MANUT|ENCARREG|\bENC\b|SUPERV|LIDER|ENGENH|TEC.*SEGUR|PLANEJ|ORCAMENT|PROJETIST|ALMOXAR|MOTORIST|OP\.? DE MAQ|OPERADOR DE MAQ|VIGILANT|JARDIN|SERV(?:ICOS)? GERAIS|AUX.*LIMPEZA|RECEPCION|OFICIAL|COORDENAD|RESGATIST|AUX.*ADM|ASSISTENTE ADM'
NAO_MO = r'\bSV[- ]?(?:TUBO|CABO|LUVA|TE |PISO|TELHA|VALVULA|REGISTRO|CRUZETA|CUBA|CAIXA|ELETRODUTO|LUMINARIA|PLACA|PINTURA|APLICACAO|FURACAO|JATEAMENTO|REVESTIMENTO|CONCRETO|FORMA|ARMADURA)|\bM2\b|\bM3\b|MM\b|EQUIPAMENTO|CAMINHAO|RETRO ?ESCAV|ESCAVADEIRA|VEICULO|CONTAINER|MATERIA|PICK UP|GRUA|RADIO'

def modificador(t):
    return next((k for k, p in MODS if re.search(p, t)), None)

df['mao_de_obra'] = (df['t'].str.contains(CARGOS) & df['um'].isin(['H', 'SV', 'MON', 'ST'])
                     & ~df['t'].str.contains(NAO_MO))
df['mod'] = None
df.loc[df['mao_de_obra'], 'mod'] = df.loc[df['mao_de_obra'], 't'].map(modificador)

ESPERADA = {'HN':'H','HE-SS':'H','HE-DF':'H','HEN-SS':'H','HEN-DF':'H','AD-NOT':'H','PER-H':'H','INS-H':'H',
            'PER-M':'MON','POSTO-M':'MON','MOB':'SV'}
df['um_esperada'] = df['mod'].map(ESPERADA)

mo_df = df[df['mao_de_obra']]
verif = df[df['um_esperada'].notna()].copy()
verif['f_um_fora_esperado'] = verif['um'] != verif['um_esperada']
df['f_um_fora_esperado'] = verif['f_um_fora_esperado'].reindex(df.index)

print('Itens de mão de obra (aprox.):', len(mo_df))
print('Sem modificador identificado:', mo_df['mod'].isna().sum())
print('Verificáveis (modificador com unidade esperada):', len(verif))
print('Fora da unidade esperada:', verif['f_um_fora_esperado'].sum(),
      f"({verif['f_um_fora_esperado'].mean()*100:.1f}%)")

Itens de mão de obra (aprox.): 3899
Sem modificador identificado: 774
Verificáveis (modificador com unidade esperada): 3123
Fora da unidade esperada: 706 (22.6%)


In [35]:
# Desvio por modificador e por unidade cadastrada
tab = (verif.groupby('mod')['f_um_fora_esperado']
       .agg(itens='size', fora='sum', pct=lambda x: round(x.mean()*100, 1)))
display(tab)

print('Unidade cadastrada x modificador:')
display(pd.crosstab(verif['mod'], verif['um']))

print('Desvios por grupo:')
display(verif.groupby('grupo')['f_um_fora_esperado'].agg(itens='size', fora='sum'))

,itens,fora,pct
mod,,,
AD-NOT,379,70,18.5
HE-DF,240,28,11.7
HE-SS,951,304,32.0
HEN-DF,70,0,0.0
HEN-SS,306,8,2.6
HN,421,47,11.2
INS-H,229,21,9.2
MOB,122,0,0.0
PER-H,198,27,13.6


Unidade cadastrada x modificador:


um,H,MON,SV
mod,,,
AD-NOT,309,0,70
HE-DF,212,1,27
HE-SS,647,1,303
HEN-DF,70,0,0
HEN-SS,298,0,8
HN,374,0,47
INS-H,208,9,12
MOB,0,0,122
PER-H,171,14,13


Desvios por grupo:


,itens,fora
grupo,,
Civil,1247,132
Eletromecânica,1528,339
Facilities,348,235


In [36]:
# Diárias: a unidade esperada é TAG (Dia)
dia = df[df['t'].str.contains(r'\bDIARIAS?\b')]
print('Itens com "diária":', len(dia))
print(dia['um'].value_counts())

Itens com "diária": 114
um
SV     89
TAG    22
H       3
Name: count, dtype: int64


In [37]:
# Conferir amostras de desvios (por hora e mensais)
desv = verif[verif['f_um_fora_esperado']]
print('--- Esperada H, cadastrada diferente')
print(desv[desv['um_esperada'] == 'H'][['descricao', 'mod', 'um']].sample(25, random_state=11).to_string())
print('\n--- Esperada MON, cadastrada diferente')
print(desv[desv['um_esperada'] == 'MON'][['descricao', 'mod', 'um']].sample(15, random_state=11).to_string())

--- Esperada H, cadastrada diferente
                                     descricao     mod   um
6331                     Almoxarife Hr. Normal      HN   SV
4791                  Sv Soldador H Adm HE 70%   HE-SS   SV
4842               Sv Eletricista H Esp HE 50%   HE-SS   SV
7721  SV - JARDINAGEM - Op Roçadeira periculos   PER-H   SV
4822                 Sv Ajudante H Esp HE 100%   HE-SS   SV
5658               LIDER CALDEIRARIA - HE.100%   HE-SS   SV
8895     SV-H. EXTRA 50%(SEG-SEX)-VIGILANTE BA   HE-SS   SV
8998                  H.EXTRA -MOTORISTA  100%   HE-SS   SV
8500          SUPERVISOR DE CONTRATO - HE 100%   HE-SS   SV
6753          PERICULOSIDADE MONTADOR(A) LÍDER   PER-H   SV
7670  SV-AUX SERVIÇOS GERAIS 12H/NOITE I-INSAL   INS-H   SV
4831                  Sv Mecânico H Esp HE 50%   HE-SS   SV
8689       MO AUX. LIMPEZA PREDIAL (INSALUBRE)   INS-H   SV
3554   Caldeireiro Alpinista N1/N2 add Pericul   PER-H  MON
6754             PERICULOSIDADE ENCARREGADO(A)   PER-H   SV
481

**Como ler:** os itens por hora com unidade SV e os itens mensais com SV são desvios da regra das abas. Na revisão por amostra, os mensais foram quase todos desvios reais; entre os por hora, parte é ambígua (periculosidade ou insalubridade sem "hora" nem "mensal" na descrição).

**Limitações:** a periculosidade sobre hora extra (modificador separado na aba 3) ainda cai em hora extra, e os itens sem modificador identificável não podem ser checados.

### Validação por amostra
O que fazer:
* 20 exemplos de cada flag para conferir se são problema de verdade. 
* Reportar o percentual ajustado, não o bruto.

In [38]:
for f in ['f_generica', 'f_cod_no_nome', 'f_limite40']:
    print('\n---', f)
    print(df[df[f]][['grupo', 'descricao', 'um']].sample(min(20, df[f].sum()), random_state=1).to_string())


--- f_generica
               grupo                         descricao   um
7341      Facilities                     VEÍCULO UTMII   SV
3720           Civil                CARPINTEIRO ROTINA   SV
5933  Eletromecânica        SOLDADOR (PRERICULOSIDADE)    H
9600      Facilities                      AUX. LIMPEZA   SV
6813  Eletromecânica  ADMINISTRATIVO (PRERICULOSIDADE)  MON
9358      Facilities                    Picolé gourmet   SV
8923      Facilities                        VIDRACEIRO   SV
43             Civil                SERRALHEIRO ROTINA   SV
8698      Facilities                CAPINEIRO EVENTUAL   SV
5930  Eletromecânica          SOLDADOR (INSALUBRIDADE)    H
9574      Facilities               BOMBA PULVERIZADORA   SV
84             Civil              CARPINTEIRO - MENSAL   SV
2921           Civil                      SV LAVATÓRIO   ST
8770      Facilities                          ARTIFICE   SV
7271      Facilities                      PORTEIRO 24H   SV
9537      Facilities    

## 6. Exportar resultados (opcional)

In [19]:
with pd.ExcelWriter('diagnostico.xlsx') as xw:
    df.to_excel(xw, sheet_name='base_com_flags', index=False)
    resumo_grupo.to_excel(xw, sheet_name='resumo_grupo')
    resumo.to_excel(xw, sheet_name='resumo_flags')
    tab.to_excel(xw, sheet_name='unidade_por_modificador')
print('Arquivo diagnostico.xlsx gerado.')

Arquivo diagnostico.xlsx gerado.
